In [ ]:
# clone the repo for running evaluation
!git clone https://github.com/AI4Bharat/indicTrans.git
%cd indicTrans
# clone requirements repositories
!git clone https://github.com/anoopkunchukuttan/indic_nlp_library.git
!git clone https://github.com/anoopkunchukuttan/indic_nlp_resources.git
!git clone https://github.com/rsennrich/subword-nmt.git
%cd ..
# Install requirements
!pip install sacremoses pandas mock sacrebleu tensorboardX pyarrow indic-nlp-library
!pip install mosestokenizer subword-nmt

In [ ]:
# Install fairseq from source
!git clone https://github.com/pytorch/fairseq.git
%cd content
!git clone https://github.com/wengong-jin/fairseq-py.git
# %cd fairseq

In [ ]:
# xformers
%cd fairseq
!pip install ./
! pip install xformers
%cd ..
# download the en2indic model
!wget https://ai4b-public-nlu-nlg.objectstore.e2enetworks.net/en2indic.zip
!unzip en2indic.zip
%cd indicTrans

In [ ]:
# add fairseq folder to python path
import os
os.environ['PYTHONPATH'] += ":/content/fairseq/"
# sanity check to see if fairseq is installed
from fairseq import checkpoint_utils, distributed_utils, options, tasks, utils

In [ ]:
from indicTrans.inference.engine import Model

en2indic_model = Model(expdir='../en-indic')

In [ ]:
!pip install python-docx
!pip install ai4bharat-transliteration

import codecs
import re
from docx import Document
from ai4bharat.transliteration import XlitEngine
import os

# Constants for file paths
INPUT_DOCX_FILE = "/content/delimited_Input.docx"
OUTPUT_TXT_FILE = "/content/input_txt.txt"
TRANSLATED_TXT_FILE_PATH = "/content/output_txt.txt"
OUTPUT_DOCX_FILE = "/content/translated_output.docx"
TRANSLITERATED_DOCX_FILE = "/content/transliterated_output.docx"

# Convert DOCX to TXT
def convert_docx_to_txt(input_path, output_path):
    doc = Document(input_path)
    with open(output_path, 'w', encoding='utf-8') as txt_file:
        for paragraph in doc.paragraphs:
            txt_file.write(paragraph.text + '\n')
    print(f'Conversion completed. Text saved to {output_path}')

# Translate paragraphs and headings
def translate_text(input_path, output_path, target_language):
    with open(input_path, 'r', encoding='utf-8') as input_file:
        en_text = input_file.read()

    heading_pattern = r'# (.*?) #'
    word_pattern = r'{{(.*?)}}'
    paragraph_pattern = r'\*(.*?)\*'

    headings = re.findall(heading_pattern, en_text)
    paragraphs = re.findall(paragraph_pattern, en_text)

    out_paragraphs = [en2indic_model.translate_paragraph(p.strip(), 'en', target_language) for p in paragraphs if p.strip()]
    out_headings = [en2indic_model.translate_paragraph(h.strip(), 'en', target_language) for h in headings if h.strip()]

    max_len = max(len(out_headings), len(out_paragraphs))
    out_headings.extend([''] * (max_len - len(out_headings)))
    out_paragraphs.extend([''] * (max_len - len(out_paragraphs)))

    combined_data = "\n\n".join(f"{h}\n{p}" for h, p in zip(out_headings, out_paragraphs))

    with codecs.open(output_path, 'w', encoding='utf-8') as output_file:
        output_file.write(combined_data)
    print(f"Translation complete. Translated text saved as: {output_path}")

# Convert TXT to DOCX
def convert_txt_to_doc(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as txt_file:
        text = txt_file.read()

    doc = Document()
    paragraphs = text.split('\n\n')

    for paragraph in paragraphs:
        if paragraph.strip():
            doc.add_paragraph(paragraph.strip())

    doc.save(output_path)
    print(f"Conversion complete. Word document saved as: {output_path}")

# Transliterate sentences in DOCX
def transliterate_docx(input_path, output_path, target_language):
    doc = Document(input_path)
    e = XlitEngine([target_language], beam_width=10, src_script_type="en")

    pattern = r"\{\{([^}]*)\}\}"
    output_doc = Document()

    for paragraph in doc.paragraphs:
        new_text = paragraph.text
        matches = re.finditer(pattern, paragraph.text)
        for match in matches:
            sentence_to_translit = match.group(1)
            transliterated_sentence = e.translit_sentence(sentence_to_translit)
            if isinstance(transliterated_sentence, dict):
                transliterated_sentence = list(transliterated_sentence.values())[0]
            new_text = new_text.replace(match.group(), transliterated_sentence)
        output_doc.add_paragraph(new_text)

    output_doc.save(output_path)
    print(f"Transliteration complete. Transliterated document saved as: {output_path}")

# Main processing steps
def main():
    target_language = input("Enter language: ")

    convert_docx_to_txt(INPUT_DOCX_FILE, OUTPUT_TXT_FILE)
    translate_text(OUTPUT_TXT_FILE, TRANSLATED_TXT_FILE_PATH, target_language)
    convert_txt_to_doc(TRANSLATED_TXT_FILE_PATH, OUTPUT_DOCX_FILE)
    transliterate_docx(OUTPUT_DOCX_FILE, TRANSLITERATED_DOCX_FILE, target_language)

if __name__ == "__main__":
    main()
